# Fall Detection TinyML — Training and C Export with emlearn

This notebook implements the full training/export pipeline required for the project:

1. Read the embedded data generator (`patient_data_generator.c`).
2. Generate synthetic NORMAL and FALL windows using the same logic used by the Contiki node.
3. Extract the same 21 features used by `patient_tinyml.c`.
4. Split the dataset into train and test sets.
5. Train a small MLP model.
6. Export the trained model to C using `emlearn`.
7. Export the `StandardScaler` constants to `fall_feature_scaler.h`.

Labels:
- `0 = NORMAL`
- `1 = FALL`

## Environment

Use Python 3.11 for best compatibility with `emlearn`.

If packages are missing, run this cell once after selecting the correct kernel:

```python
%pip install numpy scikit-learn emlearn
```

In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import emlearn
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

print("Python:", sys.version)

Python: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 21.0.0 (clang-2100.0.123.102)]


## Configuration

Place this notebook in the same folder as `patient_data_generator.c`, or change `GENERATOR_C_PATH`.

In [2]:
BASE_DIR = Path(".").resolve()

GENERATOR_C_PATH = BASE_DIR / "patient_data_generator.c"

OUT_DIR = BASE_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset size
N_NORMAL = 10_000
N_FALL = 16_000

# Reproducibility
SEED = 42

# Train/test split
TEST_SIZE = 0.20

# MLP architecture
HIDDEN_UNITS = 12

# Constants
WINDOW_SIZE = 20
NUM_RAW_SIGNALS = 9
NUM_FEATURES = 21

CLASS_NORMAL = 0
CLASS_FALL = 1
CLASS_NAMES = ["NORMAL", "FALL"]

print("Base dir:", BASE_DIR)
print("Generator:", GENERATOR_C_PATH)
print("Output dir:", OUT_DIR)

Base dir: /Users/susannabaldo/Desktop/IoT/IoT Project/smart-guard/tinyml_emlearn
Generator: /Users/susannabaldo/Desktop/IoT/IoT Project/smart-guard/tinyml_emlearn/patient_data_generator.c
Output dir: /Users/susannabaldo/Desktop/IoT/IoT Project/smart-guard/tinyml_emlearn


## 1. Parse `patient_data_generator.c`

The embedded C generator contains:

- `normal_quantiles`, used for NORMAL sample generation.
- `fall_templates`, used for FALL sample generation.

This notebook extracts those arrays directly from the C file, so the training data matches the runtime simulation logic.

In [3]:
def extract_c_array_block(source: str, array_name: str) -> str:
    start_name = source.find(array_name)
    if start_name < 0:
        raise ValueError(f"Array {array_name!r} not found in C file.")

    start_eq = source.find("=", start_name)
    if start_eq < 0:
        raise ValueError(f"Array {array_name!r} has no '='.")

    start_brace = source.find("{", start_eq)
    if start_brace < 0:
        raise ValueError(f"Array {array_name!r} has no opening brace.")

    depth = 0
    for i in range(start_brace, len(source)):
        if source[i] == "{":
            depth += 1
        elif source[i] == "}":
            depth -= 1
            if depth == 0:
                return source[start_brace : i + 1]

    raise ValueError(f"Array {array_name!r} has no matching closing brace.")


def parse_ints_from_c_array(source: str, array_name: str) -> list[int]:
    block = extract_c_array_block(source, array_name)
    return [int(x) for x in re.findall(r"-?\d+", block)]


def load_generator_constants(generator_c_path: Path) -> tuple[np.ndarray, np.ndarray]:
    source = generator_c_path.read_text(encoding="utf-8")

    normal_values = parse_ints_from_c_array(source, "normal_quantiles")
    fall_values = parse_ints_from_c_array(source, "fall_templates")

    expected_normal = NUM_RAW_SIGNALS * 7
    if len(normal_values) != expected_normal:
        raise ValueError(
            f"normal_quantiles has {len(normal_values)} values, expected {expected_normal}."
        )

    if len(fall_values) % (WINDOW_SIZE * NUM_RAW_SIGNALS) != 0:
        raise ValueError(
            "fall_templates size is not divisible by "
            f"{WINDOW_SIZE * NUM_RAW_SIGNALS}."
        )

    num_fall_templates = len(fall_values) // (WINDOW_SIZE * NUM_RAW_SIGNALS)

    normal_quantiles = np.asarray(normal_values, dtype=np.int16).reshape(
        NUM_RAW_SIGNALS,
        7,
    )

    fall_templates = np.asarray(fall_values, dtype=np.int16).reshape(
        num_fall_templates,
        WINDOW_SIZE,
        NUM_RAW_SIGNALS,
    )

    return normal_quantiles, fall_templates


normal_quantiles, fall_templates = load_generator_constants(GENERATOR_C_PATH)

print("normal_quantiles shape:", normal_quantiles.shape)
print("fall_templates shape:", fall_templates.shape)

normal_quantiles shape: (9, 7)
fall_templates shape: (8, 20, 9)


## 2. Generate NORMAL and FALL windows

NORMAL windows follow the same quantile rule implemented in C:

- 60% from `[p25, p75]`
- 25% from `[p10, p90]`
- 15% from `[p01, p99]`

FALL windows are sampled from the real SisFall-derived templates and perturbed with noise:

- accelerometers: ±30 milli-g
- gyroscope: ±5 deg/s

In [4]:
def sample_normal_feature(rng: np.random.Generator, quantiles: np.ndarray) -> int:
    r = int(rng.integers(0, 100))

    if r < 60:
        low, high = int(quantiles[2]), int(quantiles[4])
    elif r < 85:
        low, high = int(quantiles[1]), int(quantiles[5])
    else:
        low, high = int(quantiles[0]), int(quantiles[6])

    return int(rng.integers(low, high + 1))


def generate_normal_window(
    rng: np.random.Generator,
    normal_quantiles: np.ndarray,
) -> np.ndarray:
    window = np.zeros((WINDOW_SIZE, NUM_RAW_SIGNALS), dtype=np.int16)

    for i in range(WINDOW_SIZE):
        for feature_id in range(NUM_RAW_SIGNALS):
            window[i, feature_id] = sample_normal_feature(
                rng,
                normal_quantiles[feature_id],
            )

    return window


def generate_fall_window(
    rng: np.random.Generator,
    fall_templates: np.ndarray,
    acc_noise_mg: int = 30,
    gyro_noise_dps: int = 5,
) -> np.ndarray:
    template_id = int(rng.integers(0, fall_templates.shape[0]))
    window = fall_templates[template_id].astype(np.int16).copy()

    # Columns:
    # 0-2 ADXL accelerometer, 3-5 gyroscope, 6-8 MMA accelerometer.
    acc_columns = [0, 1, 2, 6, 7, 8]
    gyro_columns = [3, 4, 5]

    for col in acc_columns:
        noise = rng.integers(-acc_noise_mg, acc_noise_mg + 1, size=WINDOW_SIZE)
        window[:, col] = np.clip(
            window[:, col].astype(np.int32) + noise,
            -32768,
            32767,
        ).astype(np.int16)

    for col in gyro_columns:
        noise = rng.integers(-gyro_noise_dps, gyro_noise_dps + 1, size=WINDOW_SIZE)
        window[:, col] = np.clip(
            window[:, col].astype(np.int32) + noise,
            -32768,
            32767,
        ).astype(np.int16)

    return window

## 3. Extract the 21 TinyML features

For each window:

1. Compute squared magnitude for ADXL, gyroscope, and MMA:
   `x² + y² + z²`
2. For each magnitude signal, extract 7 statistics:
   mean, variance, min, max, range, mean absolute difference, maximum absolute difference.

Total:

`7 ADXL + 7 gyroscope + 7 MMA = 21 features`

In [5]:
def square_magnitude(x: np.ndarray, y: np.ndarray, z: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    y = y.astype(np.float32)
    z = z.astype(np.float32)
    return x * x + y * y + z * z


def calculate_signal_features(signal: np.ndarray) -> np.ndarray:
    signal = signal.astype(np.float32)

    mean = np.mean(signal, dtype=np.float32)
    variance = np.mean((signal - mean) * (signal - mean), dtype=np.float32)
    min_value = np.min(signal)
    max_value = np.max(signal)

    if signal.size > 1:
        diffs = np.abs(np.diff(signal))
        mean_abs_diff = np.mean(diffs, dtype=np.float32)
        max_abs_diff = np.max(diffs)
    else:
        mean_abs_diff = np.float32(0.0)
        max_abs_diff = np.float32(0.0)

    return np.asarray(
        [
            mean,
            variance,
            min_value,
            max_value,
            max_value - min_value,
            mean_abs_diff,
            max_abs_diff,
        ],
        dtype=np.float32,
    )


def extract_features(window: np.ndarray) -> np.ndarray:
    if window.shape != (WINDOW_SIZE, NUM_RAW_SIGNALS):
        raise ValueError(f"Invalid window shape: {window.shape}")

    adxl_magnitude = square_magnitude(window[:, 0], window[:, 1], window[:, 2])
    gyro_magnitude = square_magnitude(window[:, 3], window[:, 4], window[:, 5])
    mma_magnitude = square_magnitude(window[:, 6], window[:, 7], window[:, 8])

    features = np.zeros(NUM_FEATURES, dtype=np.float32)
    features[0:7] = calculate_signal_features(adxl_magnitude)
    features[7:14] = calculate_signal_features(gyro_magnitude)
    features[14:21] = calculate_signal_features(mma_magnitude)

    return features

## 4. Build the dataset

This cell creates:

- `X_features.npy`
- `y_labels.npy`

In [6]:
def build_dataset(
    normal_quantiles: np.ndarray,
    fall_templates: np.ndarray,
    n_normal: int,
    n_fall: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)

    total = n_normal + n_fall
    X = np.zeros((total, NUM_FEATURES), dtype=np.float32)
    y = np.zeros((total,), dtype=np.int32)

    row = 0

    for _ in range(n_normal):
        window = generate_normal_window(rng, normal_quantiles)
        X[row] = extract_features(window)
        y[row] = CLASS_NORMAL
        row += 1

    for _ in range(n_fall):
        window = generate_fall_window(rng, fall_templates)
        X[row] = extract_features(window)
        y[row] = CLASS_FALL
        row += 1

    permutation = rng.permutation(total)
    return X[permutation], y[permutation]


X, y = build_dataset(
    normal_quantiles=normal_quantiles,
    fall_templates=fall_templates,
    n_normal=N_NORMAL,
    n_fall=N_FALL,
    seed=SEED,
)

np.save(OUT_DIR / "X_features.npy", X)
np.save(OUT_DIR / "y_labels.npy", y)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("NORMAL:", int((y == CLASS_NORMAL).sum()))
print("FALL:", int((y == CLASS_FALL).sum()))
print("Saved:", OUT_DIR / "X_features.npy")
print("Saved:", OUT_DIR / "y_labels.npy")

X shape: (26000, 21)
y shape: (26000,)
NORMAL: 10000
FALL: 16000
Saved: /Users/susannabaldo/Desktop/IoT/IoT Project/smart-guard/tinyml_emlearn/X_features.npy
Saved: /Users/susannabaldo/Desktop/IoT/IoT Project/smart-guard/tinyml_emlearn/y_labels.npy


## 5. Train/test split

The dataset is split as follows:

- 80% training
- 20% test

`stratify=y` preserves the NORMAL/FALL class proportion in both sets.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train NORMAL/FALL:", int((y_train == CLASS_NORMAL).sum()), int((y_train == CLASS_FALL).sum()))
print("y_test NORMAL/FALL:", int((y_test == CLASS_NORMAL).sum()), int((y_test == CLASS_FALL).sum()))

X_train: (20800, 21)
X_test: (5200, 21)
y_train NORMAL/FALL: 8000 12800
y_test NORMAL/FALL: 2000 3200


## 6. Normalize features with StandardScaler

The model is trained on scaled features. Therefore, the same scaling constants must be exported to C and applied inside `patient_tinyml.c`.

In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print("Scaler mean shape:", scaler.mean_.shape)
print("Scaler scale shape:", scaler.scale_.shape)

Scaler mean shape: (21,)
Scaler scale shape: (21,)


## 7. Train the TinyML model

We train a small MLP:

- 21 input features
- 1 hidden layer with 12 neurons
- 2 output classes: NORMAL and FALL

In [9]:
model = MLPClassifier(
    hidden_layer_sizes=(HIDDEN_UNITS,),
    activation="relu",
    solver="adam",
    max_iter=800,
    random_state=SEED,
)

model.fit(X_train_scaled, y_train)

if list(model.classes_) != [CLASS_NORMAL, CLASS_FALL]:
    raise RuntimeError(
        f"Unexpected class order {model.classes_}. "
        "The C code assumes outputs[0]=NORMAL and outputs[1]=FALL."
    )

print("Training completed.")
print("Model classes:", model.classes_)

Training completed.
Model classes: [0 1]


## 8. Evaluate the model

Rows are true labels. Columns are predicted labels.

In [10]:
y_pred = model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred, labels=[CLASS_NORMAL, CLASS_FALL])
report = classification_report(
    y_test,
    y_pred,
    labels=[CLASS_NORMAL, CLASS_FALL],
    target_names=CLASS_NAMES,
)

print("Confusion matrix [rows=true, columns=predicted]:")
print(cm)
print()
print(report)

report_text = (
    "Confusion matrix [rows=true, columns=predicted]:\n"
    + str(cm)
    + "\n\n"
    + report
)

(OUT_DIR / "fall_training_report.txt").write_text(report_text)
print("Saved:", OUT_DIR / "fall_training_report.txt")

Confusion matrix [rows=true, columns=predicted]:
[[2000    0]
 [   0 3200]]

              precision    recall  f1-score   support

      NORMAL       1.00      1.00      1.00      2000
        FALL       1.00      1.00      1.00      3200

    accuracy                           1.00      5200
   macro avg       1.00      1.00      1.00      5200
weighted avg       1.00      1.00      1.00      5200

Saved: /Users/susannabaldo/Desktop/IoT/IoT Project/smart-guard/tinyml_emlearn/fall_training_report.txt


## 9. Export model to C with emlearn

This cell generates:

- `fall_model.h`: neural network exported by emlearn.
- `fall_feature_scaler.h`: StandardScaler constants for C inference.

In [11]:
def c_float_array(values: np.ndarray, name: str) -> str:
    formatted = ", ".join(f"{float(v):.9g}f" for v in values)
    return f"static const float {name}[{len(values)}] = {{\n  {formatted}\n}};"


def write_scaler_header(path: Path, scaler: StandardScaler) -> None:
    content = f"""#ifndef FALL_FEATURE_SCALER_H_
#define FALL_FEATURE_SCALER_H_

/*
 * Auto-generated by Fall_TinyML_Training_Export_emlearn.ipynb.
 * These values must match the StandardScaler used before emlearn export.
 */

#define FALL_NUM_FEATURES {NUM_FEATURES}

{c_float_array(scaler.mean_, "FALL_FEATURE_MEAN")}

{c_float_array(scaler.scale_, "FALL_FEATURE_SCALE")}

#endif /* FALL_FEATURE_SCALER_H_ */
"""
    path.write_text(content)


cmodel = emlearn.convert(model)

fall_model_path = OUT_DIR / "fall_model.h"
fall_scaler_path = OUT_DIR / "fall_feature_scaler.h"

cmodel.save(file=str(fall_model_path), name="fall_model")
write_scaler_header(fall_scaler_path, scaler)

print("Saved:", fall_model_path)
print("Saved:", fall_scaler_path)

Saved: /Users/susannabaldo/Desktop/IoT/IoT Project/smart-guard/tinyml_emlearn/fall_model.h
Saved: /Users/susannabaldo/Desktop/IoT/IoT Project/smart-guard/tinyml_emlearn/fall_feature_scaler.h


## 10. Files to copy into the Contiki project

Copy these generated files into the same folder as `patient_tinyml.c`:

- `fall_model.h`
- `fall_feature_scaler.h`

Then make sure `patient_tinyml.c` includes:

```c
#include "fall_model.h"
#include "fall_feature_scaler.h"
#include "eml_net.h"
```

The old `fall_binary_model.h` is no longer needed.